# 02 — Add PK, UNIQUE & FK Constraints

Declares informational constraints on our Delta tables and verifies them
via `information_schema.table_constraints`.

In [ ]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "pk_unique_demo")

## Add Constraints

In [ ]:
constraints = [
    f"ALTER TABLE {UC_CATALOG}.{UC_SCHEMA}.customers ADD CONSTRAINT customers_pk PRIMARY KEY (customer_id)",
    f"ALTER TABLE {UC_CATALOG}.{UC_SCHEMA}.customers ADD CONSTRAINT customers_email_unique UNIQUE (email)",
    f"ALTER TABLE {UC_CATALOG}.{UC_SCHEMA}.orders ADD CONSTRAINT orders_pk PRIMARY KEY (order_id)",
    f"ALTER TABLE {UC_CATALOG}.{UC_SCHEMA}.orders ADD CONSTRAINT orders_customer_fk FOREIGN KEY (customer_id) REFERENCES {UC_CATALOG}.{UC_SCHEMA}.customers(customer_id)",
]

for stmt in constraints:
    print(f"Running: {stmt}")
    spark.sql(stmt)
    print("  ✓ Done\n")

## Verify Constraints in information_schema

In [ ]:
spark.sql(f"""
    SELECT constraint_name, constraint_type, table_name, enforced
    FROM {UC_CATALOG}.information_schema.table_constraints
    WHERE constraint_schema = '{UC_SCHEMA}'
    ORDER BY table_name, constraint_type
""").show(truncate=False)

Notice the `enforced` column — these constraints are **NOT enforced** by the engine.
They exist purely as metadata for the query optimizer and for documentation.

In the next notebook, we'll prove this by inserting data that violates every one of them.